# Phase 3 — Working with Real, Messy Data (Addis Ababa Edition)

Tutorial data is clean on purpose. Real data — OSM exports, government CSVs, API responses — never is. This notebook practices the specific messiness you'll hit in your actual geo projects.

**Sections:**
1. Inconsistent column names & whitespace
2. Mixed types within one column
3. `apply()` + `lambda` for custom row logic
4. Regular expressions (`re`) for cleaning text
5. JSON handling (OSM/API-shaped data)
6. Checkpoint — clean a fully messy dataset end-to-end


## 1. Inconsistent column names & whitespace

Real exports often have column names with inconsistent capitalization, stray spaces, or symbols. This breaks code that expects exact names like `"population"` if the real header is `" Population "`.

In [28]:
import pandas as pd
import numpy as np
import re
import json

messy_columns_df = pd.DataFrame({
    " Sub-City ": ["Bole", "Yeka", "Kirkos"],
    "Population(2024)": [328900, 401000, 235100],
    "Area_KM2": [122.8, 85.9, 14.6]
})

print("Original columns:", list(messy_columns_df.columns))

Original columns: [' Sub-City ', 'Population(2024)', 'Area_KM2']


### Exercise 1.1
Clean up `messy_columns_df`'s column names: strip whitespace, lowercase everything, replace spaces/parentheses/special characters with underscores, and remove trailing underscores. End result should look like: `sub-city`, `population2024`, `area_km2` (exact cleanup style is up to you, as long as it's consistent and readable).

Tip: `df.columns = [clean_function(c) for c in df.columns]`, and `re.sub(r'[^a-z0-9]+', '_', c)` is a handy pattern once lowercased.

In [29]:
# Your code here
def clean_col(col):
    col = col.strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)   # replace any run of non-alphanumeric chars with _
    col = col.strip("_")                     # remove leading/trailing underscores
    return col

messy_columns_df.columns = [clean_col(c) for c in messy_columns_df.columns]
print(messy_columns_df.columns.tolist())

['sub_city', 'population_2024', 'area_km2']


<details>
<summary>Solution 1.1</summary>

```python
def clean_col(col):
    col = col.strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)   # replace any run of non-alphanumeric chars with _
    col = col.strip("_")                     # remove leading/trailing underscores
    return col

messy_columns_df.columns = [clean_col(c) for c in messy_columns_df.columns]
print(messy_columns_df.columns.tolist())
```
</details>

## 2. Mixed types within one column

This happens constantly with real exports: a numeric column with a stray `"N/A"`, `"unknown"`, or empty string mixed in, which forces the whole column to be read as `object` (text) instead of numbers.

In [30]:
mixed_df = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Lideta", "Arada"],
    "daily_boardings": ["1200", "950", "N/A", "600", ""]
})

print(mixed_df.dtypes)   # daily_boardings is 'object' — text, not numbers
print(mixed_df)

sub_city           str
daily_boardings    str
dtype: object
  sub_city daily_boardings
0     Bole            1200
1     Yeka             950
2   Kirkos             N/A
3   Lideta             600
4    Arada                


### Exercise 2.1
Convert `daily_boardings` to a proper numeric column, turning `"N/A"` and `""` into actual missing values (`NaN`) rather than crashing or silently becoming 0.

Tip: `pd.to_numeric(series, errors="coerce")` converts what it can and turns anything unparseable into `NaN` automatically.

In [31]:
# Your code here
mixed_df["daily_boardings"] = pd.to_numeric(mixed_df["daily_boardings"], errors="coerce")
print(mixed_df.dtypes)
print(mixed_df)

sub_city               str
daily_boardings    float64
dtype: object
  sub_city  daily_boardings
0     Bole           1200.0
1     Yeka            950.0
2   Kirkos              NaN
3   Lideta            600.0
4    Arada              NaN


<details>
<summary>Solution 2.1</summary>

```python
mixed_df["daily_boardings"] = pd.to_numeric(mixed_df["daily_boardings"], errors="coerce")
print(mixed_df.dtypes)
print(mixed_df)
```

`errors="coerce"` is the key part — without it, `pd.to_numeric` would raise an error the moment it hits `"N/A"`. With it, unparseable values just become `NaN`, which you can then handle with `.fillna()` or `.dropna()` as appropriate.
</details>

## 3. `apply()` + `lambda` for custom row logic

A `lambda` is just a one-line anonymous function — useful for quick logic inside `.apply()` that doesn't deserve its own named function. Syntax: `lambda x: <expression using x>`.

In [34]:
sub_city_names = pd.Series(["Bole", "Nifas Silk-Lafto", "Addis Ketema", "Akaky Kaliti"])

# Using apply + lambda to build a URL-safe slug
slugs = sub_city_names.apply(lambda name: name.lower().replace(" ", "-"))
print(slugs)

0                bole
1    nifas-silk-lafto
2        addis-ketema
3        akaky-kaliti
dtype: str


### Exercise 3.1
Given the Series `raw_names` below (with inconsistent spacing and capitalization), use `apply()` with a `lambda` to produce a clean "Title Case" version of each name with extra internal whitespace collapsed to a single space.

```python
raw_names = pd.Series(["  bole", "YEKA  ", "kir kos", "Lideta"])
```

Tip: `" ".join(s.split())` collapses any amount of whitespace (including leading/trailing) into single spaces — a very common cleaning trick.

In [53]:
raw_names = pd.Series(["  bole", "YEKA  ", "kir kos", "Lideta"])

# Your code here
clean_names = raw_names.apply(lambda s: " ".join(s.split()).title())
print(clean_names)

0       Bole
1       Yeka
2    Kir Kos
3     Lideta
dtype: str


<details>
<summary>Solution 3.1</summary>

```python
clean_names = raw_names.apply(lambda s: " ".join(s.split()).title())
print(clean_names)
```
</details>

## 4. Regular expressions — cleaning text with patterns

Regex lets you match *patterns* in text, not just exact strings. You won't need to become a regex expert, but a handful of patterns cover 90% of real cleaning needs:

- `\d+` — one or more digits
- `[^a-zA-Z0-9]` — anything that ISN'T a letter or digit
- `re.sub(pattern, replacement, text)` — find-and-replace using a pattern
- `re.findall(pattern, text)` — extract all matches as a list

In [54]:
# Extracting numbers embedded in messy text (common in OSM tags, e.g. "capacity=~50 people")
raw_capacity_strings = [
    "capacity: ~50 people",
    "cap. 120",
    "unknown",
    "approx 75-80 riders"
]

for s in raw_capacity_strings:
    numbers_found = re.findall(r"\d+", s)
    print(f"'{s}' -> {numbers_found}")

'capacity: ~50 people' -> ['50']
'cap. 120' -> ['120']
'unknown' -> []
'approx 75-80 riders' -> ['75', '80']


### Exercise 4.1
Given the list below, extract just the **first** number found in each string as an integer (use `None` if no number exists at all). These represent messy "stop capacity" notes you might actually pull from OSM tags.

```python
capacity_notes = [
    "seats: 40",
    "no data",
    "~65 standing",
    "120-150 depending on route"
]
```

In [57]:
capacity_notes = [
    "seats: 40",
    "no data",
    "~65 standing",
    "120-150 depending on route"
]

# Your code here
cleaned_capacities = []
for note in capacity_notes:
    numbers = re.findall(r"\d+", note)
    if numbers:
        cleaned_capacities.append(int(numbers[0]))
    else:
        cleaned_capacities.append(None)

print(cleaned_capacities)

[40, None, 65, 120]


<details>
<summary>Solution 4.1</summary>

```python
cleaned_capacities = []
for note in capacity_notes:
    numbers = re.findall(r"\d+", note)
    if numbers:
        cleaned_capacities.append(int(numbers[0]))
    else:
        cleaned_capacities.append(None)

print(cleaned_capacities)
```
</details>

## 5. JSON handling

OSM's Overpass API and most web APIs return JSON — nested dictionaries and lists. `osmnx` parses this for you normally, but understanding the raw shape helps when something doesn't come through as expected.

In [58]:
# A simplified example of what an Overpass API response element might look like
sample_osm_json = '''
{
  "elements": [
    {
      "type": "node",
      "id": 123456,
      "lat": 9.0192,
      "lon": 38.7525,
      "tags": {
        "highway": "bus_stop",
        "name": "Bole Bus Stop",
        "shelter": "yes"
      }
    },
    {
      "type": "node",
      "id": 123457,
      "lat": 9.0250,
      "lon": 38.7600,
      "tags": {
        "highway": "bus_stop",
        "name": "Meskel Square"
      }
    }
  ]
}
'''

parsed = json.loads(sample_osm_json)
print(type(parsed))
print(parsed["elements"][0]["tags"]["name"])

<class 'dict'>
Bole Bus Stop


### Exercise 5.1
Parse `sample_osm_json` (above) and build a Pandas DataFrame with columns `name`, `lat`, `lon`, `has_shelter` (True/False, based on whether the `shelter` tag exists and equals `"yes"`) — one row per element. Handle the case where `"name"` or `"shelter"` might be missing from `tags` (use `.get()` on the dict instead of `[]` indexing, which would crash on a missing key).

In [59]:
# Your code here
rows = []
for element in parsed["elements"]:
    tags = element.get("tags", {})
    rows.append({
        "name": tags.get("name", "Unknown"),
        "lat": element["lat"],
        "lon": element["lon"],
        "has_shelter": tags.get("shelter") == "yes"
    })

stops_from_json = pd.DataFrame(rows)
print(stops_from_json)

            name     lat      lon  has_shelter
0  Bole Bus Stop  9.0192  38.7525         True
1  Meskel Square  9.0250  38.7600        False


<details>
<summary>Solution 5.1</summary>

```python
rows = []
for element in parsed["elements"]:
    tags = element.get("tags", {})
    rows.append({
        "name": tags.get("name", "Unknown"),
        "lat": element["lat"],
        "lon": element["lon"],
        "has_shelter": tags.get("shelter") == "yes"
    })

stops_from_json = pd.DataFrame(rows)
print(stops_from_json)
```

`.get("key", default)` is the key habit here — it never crashes on a missing key, unlike `tags["shelter"]` which would raise a `KeyError` the moment one element doesn't have that tag (very common in real OSM data, since tagging is inconsistent).
</details>

## 6. Checkpoint — clean a fully messy dataset end-to-end

Below is one deliberately messy dataset combining every problem from this notebook: bad column names, mixed types, extra whitespace, and embedded text needing regex extraction.

**Task:** clean it into a usable DataFrame with:
- Clean column names (`sub_city`, `daily_boardings`, `capacity`)
- `sub_city` names properly title-cased with whitespace collapsed
- `daily_boardings` as a proper numeric column (`N/A`/blank → `NaN`, then filled with the column mean)
- `capacity` as an integer extracted from the messy text notes (`None` if unextractable, then dropped — rows with no capacity info aren't usable for this analysis)

Do this in one clean sequence of steps, commenting briefly what each step does.

In [63]:
messy_final_df = pd.DataFrame({
    " Sub-City Name ": ["  bole", "YEKA  ", "kir kos", "Lideta", " addis ketema"],
    "Boardings(daily)": ["1200", "N/A", "875", "", "640"],
    "capacity_notes": ["seats: 40", "~65 standing", "no data", "120-150 depending", "cap. 55"]
})

# Your code here
df = messy_final_df.copy()

# 1. Clean column names
def clean_col(col):
    col = col.strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")

df.columns = [clean_col(c) for c in df.columns]
# columns are now something like: sub_city_name, boardings_daily, capacity_notes
df = df.rename(columns={"sub_city_name": "sub_city", "boardings_daily": "daily_boardings"})

# 2. Clean sub_city text
df["sub_city"] = df["sub_city"].apply(lambda s: " ".join(s.split()).title())

# 3. Fix daily_boardings: coerce to numeric, fill NaN with column mean
df["daily_boardings"] = pd.to_numeric(df["daily_boardings"], errors="coerce")
df["daily_boardings"] = df["daily_boardings"].fillna(df["daily_boardings"].mean())

# 4. Extract capacity via regex, drop rows where it's unextractable
def extract_capacity(note):
    numbers = re.findall(r"\d+", note)
    return int(numbers[0]) if numbers else None

df["capacity"] = df["capacity_notes"].apply(extract_capacity)
df = df.drop(columns=["capacity_notes"])
df = df.dropna(subset=["capacity"])

print(df)

       sub_city  daily_boardings  capacity
0          Bole           1200.0      40.0
1          Yeka            905.0      65.0
3        Lideta            905.0     120.0
4  Addis Ketema            640.0      55.0


<details>
<summary>Reference solution</summary>

```python
df = messy_final_df.copy()

# 1. Clean column names
def clean_col(col):
    col = col.strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")

df.columns = [clean_col(c) for c in df.columns]
# columns are now something like: sub_city_name, boardings_daily, capacity_notes
df = df.rename(columns={"sub_city_name": "sub_city", "boardings_daily": "daily_boardings"})

# 2. Clean sub_city text
df["sub_city"] = df["sub_city"].apply(lambda s: " ".join(s.split()).title())

# 3. Fix daily_boardings: coerce to numeric, fill NaN with column mean
df["daily_boardings"] = pd.to_numeric(df["daily_boardings"], errors="coerce")
df["daily_boardings"] = df["daily_boardings"].fillna(df["daily_boardings"].mean())

# 4. Extract capacity via regex, drop rows where it's unextractable
def extract_capacity(note):
    numbers = re.findall(r"\d+", note)
    return int(numbers[0]) if numbers else None

df["capacity"] = df["capacity_notes"].apply(extract_capacity)
df = df.drop(columns=["capacity_notes"])
df = df.dropna(subset=["capacity"])

print(df)
```
</details>

## Next steps

Phase 3 done means you're ready to go back to **Phase 4 — Geospatial Python**, reinforcing what you've already built in your `addis-transit-access` project with the CRS/geometry/spatial-join concepts underneath it. Since you've already run that project once, this phase will mostly be about understanding *why* the code you already wrote works, not learning new syntax from scratch.
